# Proyecto Integrador - IA inmobiliaria
**Nombre:** Marielena Vásquez

**Tema:** Análisis Exploratorio de Datos (EDA) y pronóstico de ventas.

**Parte 1: entrenamiento del Random Forest en Google Colab**

Flujo completo del sistema:

`Telegram -> Mistral (Ollama) -> Random Forest -> Telegram`

En este notebook construimos la pieza del medio: el modelo que estima el precio.

**Datos:** dataset publico *house_prices* (Ames Housing), OpenML ID 42165.

**Variables que usamos:**

| Dato del usuario | Variable original | Variable del modelo |
|---|---|---|
| Sector | Neighborhood | Neighborhood |
| Metros cuadrados | GrLivArea (ft2) | Area_m2 |
| Anios | YrSold - YearBuilt | Antiguedad |
| Precio (lo que predecimos) | SalePrice | SalePrice |

> Nota academica: el resultado es una estimacion educativa con datos historicos de Ames, Iowa.
> No es una tasacion profesional ni refleja precios actuales de Ecuador.

Ejecuta las celdas **en orden**, de arriba hacia abajo.

## 1. Instalar e importar librerias

In [ ]:
!pip -q install -U scikit-learn pandas joblib

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)

**Que hace cada libreria**

- `pandas`: maneja los datos en forma de tabla.
- `scikit-learn`: trae el algoritmo Random Forest y las herramientas de entrenamiento.
- `joblib`: guarda el modelo entrenado en un archivo `.pkl` para usarlo despues en Windows.
- `matplotlib`: dibuja los graficos que iran en la presentacion.

## 2. Descargar el dataset desde OpenML

In [ ]:
datos_openml = fetch_openml(data_id=42165, as_frame=True)
df = datos_openml.frame.copy()

print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
df.head()

`fetch_openml` descarga el dataset directamente desde internet, asi que no hace falta
buscar ni subir ningun CSV a mano.

## 3. Seleccionar solo las variables que necesitamos

In [ ]:
columnas = ['Neighborhood', 'GrLivArea', 'YearBuilt', 'YrSold', 'SalePrice']
df_modelo = df[columnas].copy()
df_modelo.head()

El dataset original tiene 80 columnas. Usamos solo 4 entradas para mantener el ejercicio
simple y concentrarnos en entender el flujo completo, no en exprimir la precision.

## 4. Limpieza y preparacion de los datos

In [ ]:
for columna in ['GrLivArea', 'YearBuilt', 'YrSold', 'SalePrice']:
    df_modelo[columna] = pd.to_numeric(df_modelo[columna], errors='coerce')

df_modelo = df_modelo.dropna(subset=columnas).copy()

df_modelo['Area_m2'] = df_modelo['GrLivArea'] * 0.092903
df_modelo['Antiguedad'] = df_modelo['YrSold'] - df_modelo['YearBuilt']

df_modelo = df_modelo[df_modelo['Antiguedad'] >= 0].copy()

print('Filas despues de la limpieza:', len(df_modelo))
df_modelo[['Neighborhood', 'Area_m2', 'Antiguedad', 'SalePrice']].head()

**Que hace cada linea (para explicarlo en la defensa)**

- `pd.to_numeric(..., errors='coerce')`: convierte a numero; lo que no se puede convertir queda como vacio.
- `dropna`: elimina las filas incompletas en las columnas que nos interesan.
- `* 0.092903`: convierte pies cuadrados a metros cuadrados (1 ft2 = 0.092903 m2).
- `YrSold - YearBuilt`: antiguedad de la casa al momento de la venta.
- `Antiguedad >= 0`: descartamos antiguedades negativas porque son datos inconsistentes.

In [ ]:
# Resumen rapido de los datos limpios (util para la diapositiva del dataset)
df_modelo[['Area_m2', 'Antiguedad', 'SalePrice']].describe().round(1)

### Extra: como se ven los datos

In [ ]:
figura, ejes = plt.subplots(1, 3, figsize=(16, 4))

ejes[0].hist(df_modelo['SalePrice'], bins=40, color='#8C3B32', edgecolor='white')
ejes[0].set_title('Distribucion del precio (USD)')
ejes[0].set_xlabel('SalePrice')

ejes[1].scatter(df_modelo['Area_m2'], df_modelo['SalePrice'], s=8, alpha=0.4, color='#8C3B32')
ejes[1].set_title('Area vs Precio')
ejes[1].set_xlabel('Area_m2')
ejes[1].set_ylabel('SalePrice')

precio_por_sector = df_modelo.groupby('Neighborhood')['SalePrice'].median().sort_values()
ejes[2].barh(precio_por_sector.index, precio_por_sector.values, color='#5C7A6A')
ejes[2].set_title('Precio mediano por sector')
ejes[2].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()

El tercer grafico es el mas importante para justificar el proyecto: el **sector cambia
mucho el precio**, incluso con la misma area. Por eso lo incluimos como variable.

## 5. Definir X (entradas) e y (respuesta)

In [ ]:
X = df_modelo[['Neighborhood', 'Area_m2', 'Antiguedad']].copy()
y = df_modelo['SalePrice'].copy()

print('X:', X.shape)
print('y:', y.shape)
X.head()

- **X** = lo que el modelo conoce (sector, area, antiguedad).
- **y** = lo que queremos que aprenda a estimar (el precio).

Como `y` es un numero continuo, este es un problema de **regresion** (no de clasificacion).

## 6. Separar entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print('Entrenamiento:', len(X_train))
print('Prueba:', len(X_test))

El 80% se usa para aprender y el 20% se guarda para evaluar. Evaluamos con datos que el
modelo **nunca vio**; si evaluaramos con los mismos datos de entrenamiento, el resultado
seria enganosamente bueno.

`random_state=42` hace que la division sea siempre la misma, para poder repetir el experimento.

## 7. Convertir el sector a numeros (One Hot Encoding)

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ('sector', OneHotEncoder(handle_unknown='ignore'), ['Neighborhood']),
        ('numericas', 'passthrough', ['Area_m2', 'Antiguedad'])
    ]
)

Random Forest trabaja con numeros, no con texto. `OneHotEncoder` convierte cada sector en
una columna de 0 y 1.

Usamos One Hot y no un numero por sector porque los sectores **no tienen orden**: si a
`OldTown` le pusieramos 1 y a `Gilbert` 9, el modelo asumiria que Gilbert "vale mas" solo
por el numero.

`handle_unknown='ignore'` evita que el programa se caiga si llega un sector desconocido.
Las columnas numericas pasan tal cual con `passthrough`.

## 8. Crear y entrenar el Random Forest

In [ ]:
random_forest = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

modelo = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('random_forest', random_forest)
])

modelo.fit(X_train, y_train)
print('Modelo entrenado.')

**Random Forest en simple:** construye muchos arboles de decision, cada uno entrenado con
una parte distinta de los datos. Cada arbol da su propia estimacion y el bosque promedia
todas. Varias opiniones juntas se equivocan menos que una sola.

- `n_estimators=80`: 80 arboles.
- `max_depth=10`: cada arbol no puede hacer mas de 10 preguntas encadenadas (evita memorizar).
- `min_samples_leaf=2`: cada hoja necesita al menos 2 casas (evita reglas basadas en un solo caso).

El `Pipeline` une el preprocesamiento y el modelo en un solo objeto. Asi, cuando guardemos
el `.pkl`, el archivo ya incluye la transformacion del sector y en Windows solo pasamos los
3 datos crudos.

## 9. Evaluar el modelo

In [ ]:
predicciones = modelo.predict(X_test)

mae = mean_absolute_error(y_test, predicciones)
rmse = mean_squared_error(y_test, predicciones) ** 0.5
r2 = r2_score(y_test, predicciones)

print(f'MAE:  ${mae:,.2f}')
print(f'RMSE: ${rmse:,.2f}')
print(f'R2:   {r2:.3f}')

**Como explicar las metricas**

- **MAE**: error promedio en dolares. Si el MAE es 25.000, en promedio la estimacion se
  aleja 25.000 USD del precio real.
- **RMSE**: parecido, pero castiga mas los errores grandes. Siempre es mayor o igual al MAE.
- **R2**: entre 0 y 1. Que parte de la variacion de los precios logra explicar el modelo.
  0.80 significa que explica el 80%.

Anota estos tres numeros: van en la diapositiva 5.

### Extra 1: comparar contra una prediccion tonta

In [ ]:
# "Modelo tonto": responder siempre el precio promedio, sin mirar los datos.
prediccion_tonta = np.full(len(y_test), y_train.mean())
mae_tonto = mean_absolute_error(y_test, prediccion_tonta)

print(f'MAE del modelo tonto (siempre el promedio): ${mae_tonto:,.0f}')
print(f'MAE del Random Forest:                      ${mae:,.0f}')
print(f'Mejora: {(1 - mae / mae_tonto) * 100:.1f}% menos error')

Esto responde a la pregunta *"y como se que el modelo sirve?"*: comparado con adivinar
siempre el promedio, el Random Forest reduce el error de forma clara.

### Extra 2: validacion cruzada

In [ ]:
puntajes = cross_val_score(modelo, X, y, cv=5, scoring='r2')
print('R2 en cada una de las 5 partes:', [round(p, 3) for p in puntajes])
print(f'R2 promedio: {puntajes.mean():.3f} (+/- {puntajes.std():.3f})')

Repetimos el experimento 5 veces cambiando que parte de los datos se usa para probar.
Si los 5 resultados son parecidos, el modelo es estable y no tuvimos suerte con una
division en particular.

### Extra 3: que variable pesa mas

In [ ]:
nombres = modelo.named_steps['preprocesamiento'].get_feature_names_out()
importancias = modelo.named_steps['random_forest'].feature_importances_

peso = pd.Series(importancias, index=nombres)
grupos = pd.Series({
    'Sector': peso[[n for n in nombres if n.startswith('sector__')]].sum(),
    'Area_m2': peso[[n for n in nombres if 'Area_m2' in n]].sum(),
    'Antiguedad': peso[[n for n in nombres if 'Antiguedad' in n]].sum(),
}).sort_values()

plt.figure(figsize=(7, 3))
plt.barh(grupos.index, grupos.values * 100, color='#8C3B32')
plt.xlabel('Importancia (%)')
plt.title('Que mira mas el Random Forest')
for i, valor in enumerate(grupos.values * 100):
    plt.text(valor + 0.6, i, f'{valor:.1f}%', va='center')
plt.tight_layout()
plt.show()

### Extra 4: precio real vs precio estimado

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predicciones, s=14, alpha=0.5, color='#8C3B32')

limite = [min(y_test.min(), predicciones.min()), max(y_test.max(), predicciones.max())]
plt.plot(limite, limite, '--', color='#5C7A6A', linewidth=2, label='Prediccion perfecta')

plt.xlabel('Precio real (USD)')
plt.ylabel('Precio estimado (USD)')
plt.title(f'Real vs estimado  |  R2 = {r2:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

Mientras mas pegados esten los puntos a la linea, mejor. Se nota que las casas mas caras
son las que peor estima: con solo 3 variables el modelo no ve piscina, garaje ni calidad
de acabados. Este es un buen punto para la diapositiva 9 (que mejorarias).

## 10. Prediccion de prueba

In [ ]:
ejemplo = pd.DataFrame([{
    'Neighborhood': 'OldTown',
    'Area_m2': 120,
    'Antiguedad': 25
}])

precio_estimado = modelo.predict(ejemplo)[0]
print(f'Precio estimado: ${precio_estimado:,.0f}')

In [ ]:
# Extra: comparar el mismo tipo de casa en varios sectores
comparacion = pd.DataFrame([
    {'Neighborhood': sector, 'Area_m2': 120, 'Antiguedad': 25}
    for sector in ['OldTown', 'NAmes', 'Gilbert', 'NridgHt', 'StoneBr']
])
comparacion['Precio_estimado'] = modelo.predict(comparacion).round(0)
comparacion

Misma casa, mismo tamano, misma antiguedad: solo cambia el sector y el precio cambia
bastante. Sirve como demostracion rapida en la presentacion.

## 11. Guardar los archivos para usarlos en Windows

In [ ]:
# Lista de sectores validos: la usara el bot para validar lo que extrae Mistral
sectores_ames = sorted(df_modelo['Neighborhood'].astype(str).unique().tolist())
print(len(sectores_ames), 'sectores:', sectores_ames)

In [ ]:
joblib.dump(modelo, 'modelo_casas.pkl')

with open('sectores_ames.json', 'w', encoding='utf-8') as archivo:
    json.dump(sectores_ames, archivo, ensure_ascii=False, indent=2)

with open('requirements_local.txt', 'w', encoding='utf-8') as archivo:
    archivo.write(f'scikit-learn=={sklearn.__version__}\n')
    archivo.write(f'pandas=={pd.__version__}\n')
    archivo.write('joblib\n')
    archivo.write('requests\n')

with open('metricas.json', 'w', encoding='utf-8') as archivo:
    json.dump({'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R2': round(r2, 4)},
              archivo, indent=2)

print('Archivos guardados.')

**Importante:** el `requirements_local.txt` se genera con las versiones exactas de Colab.
Si en tu Windows instalas versiones distintas, joblib puede mostrar una advertencia al
cargar el `.pkl`. Por eso lo instalamos con `pip install -r requirements_local.txt`.

In [ ]:
# Descargar los archivos a tu computadora (solo funciona en Google Colab)
from google.colab import files

for nombre in ['modelo_casas.pkl', 'sectores_ames.json', 'requirements_local.txt', 'metricas.json']:
    files.download(nombre)

---

## Listo

Copia estos archivos a la carpeta `proyecto_ia_inmobiliaria/` en tu Windows:

- `modelo_casas.pkl`
- `sectores_ames.json`
- `requirements_local.txt`

Despues sigue con Ollama + Mistral y el bot de Telegram.

### Preguntas de la defensa, respondidas

| Pregunta | Respuesta corta |
|---|---|
| Que es un problema de regresion? | Predecir un numero continuo (el precio), no una categoria. |
| Que variable predecimos? | `SalePrice`. |
| Que son X e y? | X = sector, area y antiguedad. y = el precio. |
| Por que dividir en entrenamiento y prueba? | Para medir el modelo con datos que nunca vio. |
| Por que One Hot Encoding? | El sector es texto sin orden; hay que volverlo numeros sin inventar jerarquia. |
| Que hace un Random Forest? | Muchos arboles de decision votan y se promedia el resultado. |
| Que significa MAE? | Error promedio en dolares. |
| Por que Mistral no calcula el precio? | Es un modelo de lenguaje: inventaria un numero sin base en los datos. El precio sale del Random Forest entrenado con casas reales. |
| Que extrae Mistral? | Sector, metros cuadrados y anios de antiguedad. |
| Que rol cumple Telegram? | Solo la interfaz: recibe el mensaje y devuelve la respuesta. |
| Por que no subir el token? | Cualquiera podria controlar el bot; por eso se pide con `getpass` y esta en `.gitignore`. |